In [ ]:
!rm -rf /content/Untitled Folder
!rm -rf /content/braille_images/
!rm -rf /content/images
!rm -rf /content/user_results/
!rm -rf /content/user_codes

In [ ]:
from google.colab import files
from google.colab.patches import cv2_imshow
from PIL import Image, ImageOps
from pathlib import Path
import cv2
import numpy as np

uploaded = files.upload()
filename = next(iter(uploaded))

BASE_DIR = Path("/content")
CODES = "binary_order"
OUTPUT_DIR = Path("/content/user_codes")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / f"{CODES}.txt"
image = cv2.imdecode(
    np.frombuffer(uploaded[filename], dtype=np.uint8),
    cv2.IMREAD_COLOR
)

if image is None:
    raise ValueError("Couldn't read the file.")

def fit_without_crop(source, target_size):
    source = ImageOps.exif_transpose(source).convert("RGBA")


    fitted = ImageOps.contain(
        source,
        target_size,
        method=Image.Resampling.LANCZOS,
    )

    result = Image.new("RGBA", target_size, "white")

    x = (target_size[0] - fitted.width) // 2
    y = (target_size[1] - fitted.height) // 2
    result.alpha_composite(fitted, dest=(x, y))

    return result.convert("RGB")


TARGET_SIZE = (440, 300)

source = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
prepared_image = fit_without_crop(source, TARGET_SIZE)

prepared_image = cv2.cvtColor(
    np.array(prepared_image),
    cv2.COLOR_RGB2BGR,
)


target_w, target_h = 44, 30
h, w = prepared_image.shape[:2]

#Resize to fill the target area completely.
scale = max(target_w / w, target_h / h)

new_w = max(target_w, round(w * scale))
new_h = max(target_h, round(h * scale))

resized = cv2.resize(
    prepared_image,
    (new_w, new_h),
    interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC
)

# Trim the excess parts based on the center.
x = (new_w - target_w) // 2
y = (new_h - target_h) // 2

cropped = resized[y:y + target_h, x:x + target_w]


gray = cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY)

# Convert to black and white using a brightness threshold of 50

_, binary = cv2.threshold(gray, 235, 255, cv2.THRESH_BINARY)

# Black (0) → 1, white (255) → 0
matrix = (binary == 0).astype(np.uint8)

h, w = matrix.shape

# Check that the width is divisible by 2 and the height by 3
if h % 3 != 0 or w % 2 != 0:
    raise ValueError("The width must be the multiple of 2 and the height must be the multiple of 3.")

# (Block rows, block columns, 3 rows within each block, 2 columns within each block)
sets = matrix.reshape(h // 3, 3, w // 2, 2).transpose(0, 2, 1, 3)

# Output row by row, starting from the top-left corner
with open(OUTPUT_FILE, "a", encoding="utf-8") as file:
  for row in range(sets.shape[0]):
      for col in range(sets.shape[1]):
          code = ''.join(map(str, sets[row, col].flatten()))
          file.write(code + '\n')

Saving Screenshot 2026-09-26 at 11.07.47 AM.png to Screenshot 2026-09-26 at 11.07.47 AM.png


In [ ]:
import csv
import re

from PIL import Image, ImageDraw

CODES = "binary_order"
RESULTS = "braille_translated"
BASE_DIR = Path("/content")
LAYOUT_FILE = Path("/content/user_codes") / f"{CODES}.txt"
OUTPUT_DIR = BASE_DIR / "user_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # <-- Creates /content/user_results
OUTPUT_FILE = OUTPUT_DIR / f"{RESULTS}.png"
IMAGE_DIR = BASE_DIR / "braille_images"

ROWS = 10
COLS = 22


GAP_X = 10
GAP_Y = 10
MARGIN = 20

def generate_braille_tiles(output_dir, cell_size=(40, 60), dot_radius=5):
    """Generates all 64 binary combination PNGs for 6-dot Braille."""
    output_dir.mkdir(parents=True, exist_ok=True)
    w, h = cell_size

    # Standard 6-dot positions (2 columns x 3 rows)
    # [dot1, dot2, dot3, dot4, dot5, dot6]
    dot_positions = [
        (w // 3, h // 4),       # Dot 1 (top-left)
        (w // 3, h // 2),       # Dot 2 (mid-left)
        (w // 3, 3 * h // 4),   # Dot 3 (bot-left)
        (2 * w // 3, h // 4),   # Dot 4 (top-right)
        (2 * w // 3, h // 2),   # Dot 5 (mid-right)
        (2 * w // 3, 3 * h // 4)# Dot 6 (bot-right)
    ]

    for i in range(64):
        # Convert integer to 6-bit binary string (e.g., 0 -> "000000", 63 -> "111111")
        code = f"{i:06b}"
        img = Image.new("RGBA", (w, h), (255, 255, 255, 255))
        draw = ImageDraw.Draw(img)

        for bit_idx, bit in enumerate(code):
            if bit == '1':
                cx, cy = dot_positions[bit_idx]
                draw.ellipse(
                    [cx - dot_radius, cy - dot_radius, cx + dot_radius, cy + dot_radius],
                    fill=(0, 0, 0, 255)
                )

        img.save(output_dir / f"{code}.png")

# Run this prior to running main()
generate_braille_tiles(IMAGE_DIR)


def main():
    with LAYOUT_FILE.open(encoding="utf-8-sig") as file:
        codes = [
            line.strip()
            for line in file
            if line.strip()
        ]

    # Check the number of codes

    if len(codes) != ROWS * COLS:
        raise ValueError(
            f"Expected {ROWS * COLS} braille codes. "
            f"Found {len(codes)}."
        )

    # Check that each code consists of exactly 6 binary digits (0 or 1)
    for line_number, code in enumerate(codes, start=1):
        if re.fullmatch(r"[01]{6}", code) is None:
            raise ValueError(
                f"Invalid code at position {line_number}: {code!r}"
            )

    # Group the 220 codes into 10 rows of 22 columns

    layout = [
        codes[start:start + COLS]
        for start in range(0, len(codes), COLS)
    ]

    # Keep the existing image loading, compositing, and saving code below.


    # Check that the layout has 10 rows and 22 columns

    if len(layout) != ROWS:
        raise ValueError(
            f"The layout must have {ROWS} rows. "
            f"Found {len(layout)}."
        )

    for row_index, row in enumerate(layout, start=1):
        if len(row) != COLS:
            raise ValueError(
                f"Row {row_index} must contain {COLS} codes. "
                f"Found {len(row)}."
            )

        for col_index, code in enumerate(row, start=1):
            if re.fullmatch(r"[01]{6}", code) is None:
                raise ValueError(
                    f"Invalid code at row {row_index}, "
                    f"column {col_index}: {code!r}"
                )

    # Load each required image only once
    images = {}
    cell_size = None

    for code in sorted({code for row in layout for code in row}):
        path = IMAGE_DIR / f"{code}.png"

        if not path.is_file():
            raise FileNotFoundError(f"Braille image not found: {path}")

        with Image.open(path) as source:
            tile = source.convert("RGBA")

        if cell_size is None:
            cell_size = tile.size
        elif tile.size != cell_size:
            raise ValueError(
                f"{path.name} has dimensions {tile.size}. "
                f"All images must have dimensions {cell_size}."
            )

        images[code] = tile

    # Create the full canvas

    cell_width, cell_height = cell_size

    canvas_width = (
        MARGIN * 2 + COLS * cell_width + (COLS - 1) * GAP_X
    )
    canvas_height = (
        MARGIN * 2 + ROWS * cell_height + (ROWS - 1) * GAP_Y
    )

    canvas = Image.new(
        "RGBA",
        (canvas_width, canvas_height),
        (255, 255, 255, 255),
    )

    # Arrange the 220 images from top to bottom and left to right
    for row_index, row in enumerate(layout):
        for col_index, code in enumerate(row):
            x = MARGIN + col_index * (cell_width + GAP_X)
            y = MARGIN + row_index * (cell_height + GAP_Y)

            canvas.alpha_composite(images[code], dest=(x, y))

    # Save as PNG
    canvas.convert("RGB").save(OUTPUT_FILE)
    print(f"Done: {ROWS * COLS} braille cells → {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

ValueError: Expected 220 braille codes. Found 660.

In [ ]:
# from PIL import Image, ImageOps

# def fit_without_crop(source, target_size):
#     source = ImageOps.exif_transpose(source).convert("RGBA")


#     fitted = ImageOps.contain(
#         source,
#         target_size,
#         method=Image.Resampling.LANCZOS,
#     )

#     result = Image.new("RGBA", target_size, "white")

#     x = (target_size[0] - fitted.width) // 2
#     y = (target_size[1] - fitted.height) // 2
#     result.alpha_composite(fitted, dest=(x, y))

#     return result.convert("RGB")

# INPUT_FILE = BASE_DIR / "/content/Screenshot 2026-09-26 at 2.10.57 PM (1).png"
# TARGET_SIZE = (440, 300)

# with Image.open(INPUT_FILE) as source:
#     prepared_image = fit_without_crop(source, TARGET_SIZE)

# prepared_image.save(Path("/content/sample_data")/ "prepared_image.png")


In [ ]:
from pathlib import Path
from PIL import Image

output_dir = Path("images")
output_dir.mkdir(parents=True, exist_ok=True)

# (x, y) coordinates for each bit — the top-left corner is (0, 0)
positions = [
    (1, 1), (3, 1),
    (1, 3), (3, 3),
    (1, 5), (3, 5),
]

for number in range(64):
    bits = f"{number:06b}"  # 000000 to 111111

    # 5 pixels wide × 7 pixels high, with a white background
    image = Image.new("RGB", (5, 7), color="white")

    for bit, position in zip(bits, positions):
        color = (0, 0, 0) if bit == "1" else (255, 255, 255)
        image.putpixel(position, color)

    output_dir = Path("/content/Untitled Folder 1")
    output_dir.mkdir(parents=True, exist_ok=True)
    image = image.resize((35, 49), resample=Image.Resampling.NEAREST)
    image.save(output_dir / f"{bits}.png")

print(f"Successfully generated 64 images: {output_dir.resolve()}")

Successfully generated 64 images: /content/Untitled Folder 1


In [ ]:
!rm -rf /content/Untitled Folder

In [ ]:
%pip install py

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

print("OpenCV version:", cv2.__version__)

OpenCV version: 5.0.0


In [ ]:
from pathlib import Path

FILE_NAME = "input_letters1"
W2B = "test_result4"

INPUT_TEXT = Path("/content/user_text") / f"{FILE_NAME}.txt"

# 1. Ensure parent directory exists and write sample text if missing
INPUT_TEXT.parent.mkdir(parents=True, exist_ok=True)
# INPUT_TEXT.write_text("Translating text...", encoding="utf-8")

# Remove any leading BOM and replace tabs with spaces
text = INPUT_TEXT.read_text(encoding="utf-8-sig").expandtabs(4)
print(text)
# English lowercase letters
letters = dict(zip(
    "abcdefghijklmnopqrstuvwxyz",
    "⠁⠃⠉⠙⠑⠋⠛⠓⠊⠚⠅⠇⠍⠝⠕⠏⠟⠗⠎⠞⠥⠧⠺⠭⠽⠵"
))

# English uppercase letters
letters.update({
    char.upper(): "⠠" + braille
    for char, braille in list(letters.items())
})

punctuation = {
    " ": " ",
    ",": "⠂",
    ".": "⠲",
    ":": "⠒",
    "\n": "\n",
    "(": "⠐⠣",
    ")": "⠐⠜",
    "[": "⠨⠣",
    "]": "⠨⠜",
    "{": "⠸⠣",
    "}": "⠸⠜",
    "′": "⠶",
    "″": "⠶⠶",
    "‴": "⠶⠶⠶",
    "=": "⠐⠶",
}

# Greek letters in UEB technical notation
greek_letters = {
    "α": "⠨⠁",  # alpha
    "β": "⠨⠃",  # beta
    "γ": "⠨⠛",  # gamma
    "δ": "⠨⠙",  # delta
    "ε": "⠨⠑",  # epsilon
    "ζ": "⠨⠵",  # zeta
    "η": "⠨⠱",  # eta
    "θ": "⠨⠹",  # theta
    "ι": "⠨⠊",  # iota
    "κ": "⠨⠅",  # kappa
    "λ": "⠨⠇",  # lambda
    "μ": "⠨⠍",  # mu
    "ν": "⠨⠝",  # nu
    "ξ": "⠨⠭",  # xi
    "ο": "⠨⠕",  # omicron
    "π": "⠨⠏",  # pi
    "ρ": "⠨⠗",  # rho
    "σ": "⠨⠎",  # sigma
    "τ": "⠨⠞",  # tau
    "υ": "⠨⠥",  # upsilon
    "φ": "⠨⠋",  # phi
    "χ": "⠨⠯",  # chi
    "ψ": "⠨⠽",  # psi
    "ω": "⠨⠺",  # omega
}

# Greek uppercase letters
greek_letters.update({
    char.upper(): "⠠" + braille
    for char, braille in list(greek_letters.items())
})

# Final sigma
greek_letters["ς"] = greek_letters["σ"]

# Digits use a numeric indicator before each number
digits = dict(zip("1234567890", "⠁⠃⠉⠙⠑⠋⠛⠓⠊⠚"))

punctuation.update({
    "+": "⠐⠖",
    "−": "⠐⠤",   # Mathematical minus (U+2212)
    "-": "⠤",     # Hyphen
    "/": "⠸⠌",   # Printed slash
    "<": "⠈⠣",
    ">": "⠈⠜",
    "'": "⠄",     # Apostrophe; use ′ for a prime
    "“": "⠦",
    "”": "⠴",
})


def to_braille(text):
    result = []
    opening_quote = True
    numeric_mode = False

    for index, char in enumerate(text):
        # Add one numeric indicator per consecutive number
        if char in digits:
            if not numeric_mode:
                result.append("⠼")
            result.append(digits[char])
            numeric_mode = True
            continue

        # A period or comma preserves numeric mode
        if numeric_mode and char in ".,":
            result.append(punctuation[char])
            continue

        # Distinguish letters a–j from digits after a number
        if numeric_mode and char in "abcdefghij":
            result.append("⠰")

        numeric_mode = False

        if char in greek_letters:
            result.append(greek_letters[char])

        elif char == '"':
            result.append("⠦" if opening_quote else "⠴")
            opening_quote = not opening_quote

        elif char in punctuation:
            result.append(punctuation[char])

        elif char in letters:
            result.append(letters[char])

        else:
            raise ValueError(
                f"Unsupported character at position {index + 1}: "
                f"{char!r} (U+{ord(char):04X})"
            )

    return "".join(result)

braille_text = to_braille(text)

output_path = (
    Path("/content/user_words2brailles") / f"{W2B}.txt"
)
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(braille_text, encoding="utf-8")

print(braille_text)
print(f"\nSaved to: {output_path}")

22. A cubic polynomial f(x) with leading coefficient 1 satisfies the following condition: There is no integer k such that f(k − 1)f(k + 1) < 0.

Given that f'(−1/4) = −1/4 and f'(1/4) < 0, find f(8). [4 points]
⠼⠃⠃⠲ ⠠⠁ ⠉⠥⠃⠊⠉ ⠏⠕⠇⠽⠝⠕⠍⠊⠁⠇ ⠋⠐⠣⠭⠐⠜ ⠺⠊⠞⠓ ⠇⠑⠁⠙⠊⠝⠛ ⠉⠕⠑⠋⠋⠊⠉⠊⠑⠝⠞ ⠼⠁ ⠎⠁⠞⠊⠎⠋⠊⠑⠎ ⠞⠓⠑ ⠋⠕⠇⠇⠕⠺⠊⠝⠛ ⠉⠕⠝⠙⠊⠞⠊⠕⠝⠒ ⠠⠞⠓⠑⠗⠑ ⠊⠎ ⠝⠕ ⠊⠝⠞⠑⠛⠑⠗ ⠅ ⠎⠥⠉⠓ ⠞⠓⠁⠞ ⠋⠐⠣⠅ ⠐⠤ ⠼⠁⠐⠜⠋⠐⠣⠅ ⠐⠖ ⠼⠁⠐⠜ ⠈⠣ ⠼⠚⠲

⠠⠛⠊⠧⠑⠝ ⠞⠓⠁⠞ ⠋⠄⠐⠣⠐⠤⠼⠁⠸⠌⠼⠙⠐⠜ ⠐⠶ ⠐⠤⠼⠁⠸⠌⠼⠙ ⠁⠝⠙ ⠋⠄⠐⠣⠼⠁⠸⠌⠼⠙⠐⠜ ⠈⠣ ⠼⠚⠂ ⠋⠊⠝⠙ ⠋⠐⠣⠼⠓⠐⠜⠲ ⠨⠣⠼⠙ ⠏⠕⠊⠝⠞⠎⠨⠜

Saved to: /content/user_words2brailles/test_result4.txt


In [ ]:
%pip install pybraille

  Preparing metadata (setup.py) ... done
  Created wheel for pybraille: filename=pybraille-1.0.0-py3-none-any.whl size=3606 sha256=256f010500dd6f915a898c4f70222195f0702786cb997d8ac93219e2d762d4a9
  Stored in directory: /root/.cache/pip/wheels/68/25/63/4a113a4d2ce339c26bd1433b9644494e10020aff1456be92ee
Successfully built pybraille
